# Lesson 6: Prompts and Context

**Time:** 45 min | **Cost:** $0 (no API calls)

---

## What is a Prompt?

Everything you send to an LLM is a **prompt**. There's no special magic -- it's text you give to the AI, and it responds based on that text.

It's like **writing a brief for a freelance writer**:

- A vague brief ("write something about SEO") gets vague, generic content back
- A detailed brief (target audience, tone, keywords, examples) gets what you need

The same applies to AI. **The better your prompt, the better the output.**

In our project, prompts show up in two places:

| In Agno code | What it is | Example |
|---|---|---|
| `instructions=[...]` | **System prompt** -- permanent behavior | "You are an expert SEO researcher" |
| `agent.run("...")` | **User prompt** -- specific task this turn | "Research the topic: link building" |

## Anatomy of a Good Prompt

A good prompt has **four components**. You don't always need all four, but the more you include, the better your results:

1. **Role** -- Who the AI should be
   - "You are an expert SEO writer with 10 years of experience"
   - This sets the tone, vocabulary, and expertise level

2. **Task** -- What to do
   - "Write a meta description for an article about on-page SEO"
   - Be specific: "Write 5 bullet points" beats "Write some content"

3. **Constraints** -- Rules to follow
   - "Maximum 160 characters", "Use active voice", "Do NOT use clickbait"
   - Constraints prevent the AI from going off-track

4. **Examples** -- Show what good output looks like (few-shot)
   - Give 1-3 examples of the quality and format you expect
   - This is **few-shot prompting** and it works remarkably well

In [ ]:
# BAD prompt -- vague, no structure
bad_prompt = "Write about SEO"

# GOOD prompt -- role, task, constraints, example
good_prompt = """You are an expert SEO content writer with 10 years of experience.

Write a meta description for an article about "on-page SEO optimization."

Constraints:
- Maximum 160 characters
- Include the keyword "on-page SEO" naturally
- Use active voice
- End with a call to action

Example of a good meta description:
"Master technical SEO with our step-by-step guide. Learn crawling, indexing, and site speed optimization. Start improving your rankings today."
"""

print(f"Bad prompt:  {len(bad_prompt)} characters")
print(f"Good prompt: {len(good_prompt)} characters")
print(f"\nThe good prompt is {len(good_prompt) // len(bad_prompt)}x longer,")
print("but it will produce MUCH better results!")
print()
print("--- Bad prompt ---")
print(bad_prompt)
print()
print("--- Good prompt ---")
print(good_prompt)

## System Prompt vs User Prompt

LLMs receive **two types of prompts** in every conversation:

**System prompt** = persistent behavior. Set once. The agent's personality and rules.
- In Agno: `instructions=["You are an SEO expert", "Answer concisely"]`
- It's a **job description** -- stays the same for every task

**User prompt** = specific task this turn. Changes each time.
- In Agno: `agent.run("What is a meta description?")`
- It's a **work request** -- different every time

Under the hood, the LLM sees both combined:

```
[System: You are an SEO expert. Answer concisely.]
[User: What is a meta description?]
   --> LLM processes both together --> Response
```

The system prompt shapes **HOW** it responds. The user prompt determines **WHAT** it responds about.

In [ ]:
# This is what happens under the hood when an agent runs

system_prompt = "You are an SEO expert. Answer concisely and clearly."
user_prompt = "What is a meta description?"

# The LLM sees both together:
full_context = f"""--- System Prompt (from instructions) ---
{system_prompt}

--- User Prompt (from agent.run()) ---
{user_prompt}
"""

print(full_context)
print("The LLM processes this entire context to generate its response.")
print("The system prompt shapes HOW it responds.")
print("The user prompt determines WHAT it responds about.")

## Prompt Engineering for SEO Teams

Five practical techniques that work well for SEO content tasks:

### 1. Specificity beats length
"Write 5 bullet points about link building benefits" is better than "Write some content about link building." Be precise about what you want.

### 2. Output format specification
Tell the AI what format you want: "Return as JSON with keys: title, description, keywords" or "Use H2 headings for each section." If you don't specify, you'll get whatever format the AI feels like.

### 3. Few-shot examples
Show 2-3 examples of what you want. The AI will match the pattern. This is one of the most powerful techniques and it's free -- you need good examples, that's it.

### 4. Negative instructions
"Do NOT use clickbait titles" or "Do NOT start with 'In today's digital landscape'" works well. LLMs are good at avoiding things you explicitly tell them not to do.

### 5. Step-by-step reasoning
"First analyze the keyword intent, then identify the target audience, then write the title" produces better results than "Write a title." Breaking tasks into steps guides the AI's thinking process.

In [ ]:
# These are the ACTUAL instructions from our Content Writer agent
# (from output/backend/agents/content_writer.py)

writer_instructions = [
    "You are an expert SEO content writer.",
    "Research the given topic using web search.",
    "Identify primary and secondary keywords, analyze what top-ranking content covers, "
    "and find content gaps.",
    "Write a comprehensive, SEO-optimized article based on your research.",
]

print("Content Writer instructions breakdown:\n")
for i, instruction in enumerate(writer_instructions, 1):
    print(f"  {i}. \"{instruction}\"")

print("\nNotice the pattern:")
print("  Line 1: ROLE       -- 'You are an expert SEO content writer'")
print("  Line 2: TASK       -- 'Research the given topic using web search'")
print("  Lines 3-4: CONSTRAINTS -- what to include, how to format")

## Prompt Templates

In production, you don't write prompts from scratch every time. You build **prompt templates** -- reusable functions that take variables and return a formatted prompt string.

This is what our pipeline agents do. The `instructions` stay the same, but the topic, keywords, and other details change with each run.

A prompt template is a **Python function** that uses **f-strings** (from Lesson 1) to insert variables into a pre-written prompt.

In [ ]:
# A reusable prompt template as a Python function

def meta_description_prompt(topic, keyword, max_chars=160):
    """Generate a prompt for writing a meta description."""
    return f"""You are an expert SEO copywriter.

Write a meta description for an article about "{topic}".

Requirements:
- Maximum {max_chars} characters
- Include the keyword "{keyword}" naturally
- Use active voice
- End with a call to action
- Make it compelling enough to click
"""

# Test it with different inputs
prompt1 = meta_description_prompt("on-page SEO", "on-page SEO optimization")
prompt2 = meta_description_prompt("link building strategies", "link building")

print("=== Prompt 1 ===")
print(prompt1)
print("=== Prompt 2 ===")
print(prompt2)
print("Same template, different inputs -- this is how production agents work!")

## Prompt Engineering = The New SEO Skill

If you're an SEO professional, you already have the mindset for prompt engineering:

| SEO Writing | Prompt Engineering |
|---|---|
| You write for **Google's algorithm** | You write for the **LLM** |
| You use **keywords** strategically | You use **instructions** strategically |
| You **test and iterate** on content | You **test and iterate** on prompts |
| Better content = higher rankings | Better prompts = better AI output |

**Better prompts --> better agent output --> less human editing --> more content at scale**

This is why prompt engineering matters for SEO teams. It's not about replacing writers -- it's about giving AI the best possible brief so the output needs minimal editing.

## Summary

What you learned:

- **Prompt** -- everything you send to an LLM (the better the brief, the better the output)
- **System prompt** (`instructions`) -- persistent behavior, like a job description
- **User prompt** (`agent.run()`) -- specific task each time, like a work request
- **4 components** of a good prompt: **Role**, **Task**, **Constraints**, **Examples**
- **Prompt templates** -- reusable Python functions that generate prompts with f-strings
- **Prompt engineering for SEO** -- the same test-and-iterate mindset you already use

**Next lesson:** Different AI models -- why some are better for certain tasks, and how we chose specific models for each step in our pipeline.

## Exercise

Create a **prompt template function** for a different SEO task. Choose one:

- `title_generator_prompt(topic, keyword)` -- generates SEO article titles
- `keyword_cluster_prompt(seed_keyword, num_clusters)` -- groups related keywords
- `content_brief_prompt(topic, audience, word_count)` -- creates a content brief

**Requirements for your template:**

1. Include all 4 components (Role, Task, Constraints, Examples)
2. Use f-string variables for the changing parts
3. Test it with at least 2 different inputs and print both prompts

In [ ]:
# Exercise: Fill in the blanks to create a prompt template function
# Choose one of the tasks above, or use title_generator_prompt as shown below.

def title_generator_prompt(topic, keyword, num_titles=5):
    """Generate a prompt for writing SEO article titles."""
    return f"""You are an expert ___ copywriter.

___: Generate {num_titles} SEO-optimized article titles about "{topic}".

Constraints:
- Each title must include the keyword "{keyword}" ___
- Keep titles under 60 characters
- Use a mix of styles: how-to, listicle, question
- Do NOT use ___

Example of good titles for "link building":
- "7 Link Building Strategies That Actually Work in 2026"
- "How to Build Quality Backlinks: A Beginner's Guide"
"""

# Test with 2 different inputs
prompt1 = title_generator_prompt("on-page SEO", "on-page SEO")
prompt2 = title_generator_prompt("email marketing", "email campaigns", num_titles=3)

print("=== Prompt 1 ===")
print(prompt1)
print("=== Prompt 2 ===")
print(prompt2)

<details>
<summary>Click to reveal answer</summary>

```python
def title_generator_prompt(topic, keyword, num_titles=5):
    return f"""You are an expert SEO copywriter.

Task: Generate {num_titles} SEO-optimized article titles about "{topic}".

Constraints:
- Each title must include the keyword "{keyword}" naturally
- Keep titles under 60 characters
- Use a mix of styles: how-to, listicle, question
- Do NOT use clickbait

Example of good titles for "link building":
- "7 Link Building Strategies That Actually Work in 2026"
- "How to Build Quality Backlinks: A Beginner's Guide"
"""
```

The blanks are: `SEO` (role), `Task` (component label), `naturally` (constraint), `clickbait` (negative instruction).
</details>

## Bonus: See a Real LLM Response

The cell below makes one quick API call using **Claude Haiku** (the cheapest model, ~$0.001 per call).

**Optional** — requires `ANTHROPIC_API_KEY` in your `.env` file. If you don't have it yet, skip this cell and come back after Lesson 7's checkpoint.

In [ ]:
# Bonus: One real API call with Claude Haiku (~$0.001)
# Skip this cell if you don't have ANTHROPIC_API_KEY set yet

import os
from dotenv import load_dotenv
load_dotenv()

if not os.getenv("ANTHROPIC_API_KEY"):
    print("Skipped — ANTHROPIC_API_KEY not set. Come back after Lesson 7's checkpoint!")
else:
    from agno.agent import Agent
    from agno.models.anthropic import Claude

    quick_agent = Agent(
        model=Claude(id="claude-haiku-4-5-20251001"),
        instructions=["You are an SEO expert. Be concise."],
    )

    response = quick_agent.run("Give me 3 quick tips for writing better meta descriptions.")
    print(response.content)
    print("\n--- That's a real AI response from your prompt! ---")